# Importações

### Bibiliotecas Padrão

In [1]:
import pandas as pd  
import numpy  as np
import json
import os
import os.path
import pickle
import sys
import concurrent.futures
import requests
import googlemaps
import matplotlib.pyplot as plt

from pandasgui                          import show
from statsmodels.tsa.statespace.sarimax import SARIMAX
from scipy.spatial                      import KDTree
from google.auth.transport.requests     import Request
from google.oauth2.credentials          import Credentials
from google_auth_oauthlib.flow          import InstalledAppFlow
from googleapiclient.discovery          import build
from googleapiclient.errors             import HttpError

# Caminho para o arquivo de credenciais

API_MAPS_SHEETS_PATH = r"D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\api_maps_sheets.json"

# Abrindo o arquivo e carregando o conteúdo
with open(API_MAPS_SHEETS_PATH, 'r') as file:
    dados = json.load(file)

# Acessando as variáveis do JSON
CREDENTIALS_PATH = r"D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\client_secret.json"
SAMPLE_SPREADSHEET_ID = dados['SAMPLE_SPREADSHEET_ID']
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']

chave_api = dados['chave_api']
gmaps = googlemaps.Client(key=chave_api)

### Bibliotecas Próprias

In [2]:
sys.path.append(r'D:\Drive\Codigos bee6\bee6_notebooks\bee6_geral')

# Agora importe as classes
from aplica_dados_geo import (
    CalculoDistanciaTransporte,
    TrataDelegacias,MesclaDadosMunicipais,
    AgregaIptuCep)

### Arquivos Locais

In [3]:
# Define the paths to search for files
paths = [r'D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Tratados\Sao Paulo',]

# Define a function to read a file
def read_file(file_path):
    if file_path.endswith('.parquet'):
        return pd.read_parquet(file_path)
    elif file_path.endswith('.csv'):
        return pd.read_csv(file_path)
    else:
        raise ValueError(f"Unsupported file format for {file_path}")

# Function to process files in a directory
def process_directory(directory):
    dataframes = {}
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        if os.path.isfile(file_path) and (filename.endswith('.csv') or filename.endswith('.parquet')):
            df = read_file(file_path)
            name, _ = os.path.splitext(filename)
            dataframes[name] = df
    return dataframes

# Create a ThreadPoolExecutor to read files in parallel
all_dataframes = {}
with concurrent.futures.ThreadPoolExecutor() as executor:
    # Use list comprehension to process directories in parallel
    futures = [executor.submit(process_directory, path) for path in paths]
    for future in concurrent.futures.as_completed(futures):
        directory_dataframes = future.result()
        all_dataframes.update(directory_dataframes)

# Define DataFrames dynamically based on file names
globals().update(all_dataframes)

### Planilhas Gsheets

In [2]:
# Função para ler múltiplas abas de uma planilha no Google Sheets
def get_spreadsheet_data(spreadsheet_id, sheet_ranges):
    creds = None
    # O arquivo token.json armazena os tokens de acesso e atualização do usuário.
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    # Se não há credenciais (ou são inválidas), deixe o usuário se autenticar.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_PATH, SCOPES)
            creds = flow.run_local_server(port=0)
        # Salve as credenciais para a próxima execução
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    try:
        # Chama a API Google Sheets
        service = build('sheets', 'v4', credentials=creds)
        sheet = service.spreadsheets()

        # Itera sobre cada aba/intervalo na lista
        for sheet_range in sheet_ranges:
            result = sheet.values().get(spreadsheetId=spreadsheet_id, range=sheet_range).execute()
            values = result.get('values', [])

            # Cria um DataFrame com os dados
            if values:
                df = pd.DataFrame(values[1:], columns=values[0])
            else:
                df = pd.DataFrame()

            # Define dinamicamente uma variável com o nome da aba
            globals()[sheet_range] = df

    except HttpError as err:
        print(f"An error occurred: {err}")

if __name__ == '__main__':
    sheet_ranges = ["dbImoveis", "dbCompradores"] 
    get_spreadsheet_data(SAMPLE_SPREADSHEET_ID, sheet_ranges)

### Modelos de Machine Learning

In [5]:
# Caminho para o arquivo pickle
model_filename = r"D:\Drive\Codigos bee6\bee6_arquivos\urban_space\Modelos\precificacao_imoveis_sp.pkl"

# Carregar o modelo e os metadados
with open(model_filename, 'rb') as file:
    model_info = pickle.load(file)

# Extrair o modelo e os metadados do dicionário carregado
xgb_model = model_info['modelo']
metadados = model_info['metadados']

# Aplica a Precificação XGB a todos os imóveis com base no Vivareal ref 11/23

### Tratamento de bases de dados

In [6]:
# Tratamento dos dados da planilha
dbimoveis = dbImoveis[['ID Imóvel','m2', 'Num Banheiros', 'Num Quartos', 'Num Suites', 'Num Vagas Est', 'Valor Condominio', 'Valor Iptu','CEP','Área Total m2','Latitude/Longitude']]
dbimoveis.columns = ['id_imovel','area_construida_m2','num_banheiros','num_quartos','num_suites','num_vagas_est','condominio_mensal','iptu_anual','cep','area_total_m2','lat_long']

# Substituir strings vazias por NaN
dbimoveis.replace('', np.nan, inplace=True)

# Para colunas que precisam ser convertidas para inteiros:
for col in ['area_construida_m2', 'num_banheiros', 'num_quartos', 'num_suites', 'num_vagas_est','area_total_m2']:
    # Primeiro, converta a coluna para numérico, forçando erros para NaN
    dbimoveis[col] = pd.to_numeric(dbimoveis[col], errors='coerce')

    # Depois, converta para inteiro, se desejado. Use 'Int64' para suportar NaNs
    dbimoveis[col] = dbimoveis[col].astype('Int64')

# Para colunas monetárias, limpar e converter para float
for col in ['condominio_mensal', 'iptu_anual']:
    # Limpar caracteres não numéricos e converter para float
    dbimoveis[col] = (dbimoveis[col].str.replace('R', '', regex=False)
                      .str.replace('$', '', regex=False).str.replace('.', '', regex=False)
                      .str.replace(',', '.', regex=False))

    # Converter para float, forçando erros para NaN
    dbimoveis[col] = pd.to_numeric(dbimoveis[col], errors='coerce')

# Para a coluna 'cep', formatar o CEP
dbimoveis['cep'] = dbimoveis['cep'].str[:5] + dbimoveis['cep'].str[5:]

C:\Users\guici\AppData\Local\Temp\ipykernel_28612\3893651109.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dbimoveis.replace('', np.nan, inplace=True)
C:\Users\guici\AppData\Local\Temp\ipykernel_28612\3893651109.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dbimoveis[col] = pd.to_numeric(dbimoveis[col], errors='coerce')
C:\Users\guici\AppData\Local\Temp\ipykernel_28612\3893651109.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats i

In [7]:
# Trata a dbCompradores
dbcompradores = dbCompradores[['Imóvel Adquirido','Data Venda Efetiva','Orçamento Disponível']]
dbcompradores.dropna(subset = 'Imóvel Adquirido', inplace = True)
dbcompradores.columns = ['id_imovel','data_venda','valor_venda']
dbcompradores['data_venda'] = pd.to_datetime(dbcompradores['data_venda'], dayfirst=True)

dbcompradores['valor_venda'] = (dbcompradores['valor_venda']
                                .str.replace('R', '', regex=False)
                                .str.replace('$', '', regex=False)
                                .str.replace('.', '', regex=False)
                                .str.replace(',', '.', regex=False)
                                .replace('', np.nan)  
                                .astype(float))

dbcompradores = dbcompradores.merge(dbimoveis, how='left', on='id_imovel')
dbcompradores['valor_venda_m2'] = dbcompradores['valor_venda']/dbcompradores['area_construida_m2']

C:\Users\guici\AppData\Local\Temp\ipykernel_28612\869229884.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dbcompradores.dropna(subset = 'Imóvel Adquirido', inplace = True)
C:\Users\guici\AppData\Local\Temp\ipykernel_28612\869229884.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dbcompradores['data_venda'] = pd.to_datetime(dbcompradores['data_venda'], dayfirst=True)
C:\Users\guici\AppData\Local\Temp\ipykernel_28612\869229884.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_i

### Aplica as classes

In [8]:
# Primeiro atribuo a Lat e Long de cada delegacia (que não dá match exato com os Distritos), depois, calculo qual 
# o distrito mais próximo dessa delegacia, a limitação é que uma delegacia pode estar na borda de um distrito,
# sendo mais relacionada as ocorrências deste distrito do que do próprio (baseado no centro do polígono do distrito)
# mas é o que tem pra hoje. Se tivermos mais de uma delegacia mapeada pra um mesmo distrito, somo os dados.
# Usarei apenas a coluna de Registros_Policiais, sem considerar os subtipos de ocorrência, para facilitar a análise
# Os dados da delegacia são de 2022, e precisam ser normalizados pela pop de cada distrito (de 2010), o que não é o ideal, mas
# pode servir de aproximação. No final, atribuo, para cada cep de SP, um registro policial com base no distrito em que o CEP está inserido
bo_distrito = TrataDelegacias(delegacias, distrito_latlong, pop_distrito_sp, ceps_sp).process()

# O codigo, com base na informação de pontos de onibus/terminais/estacao de trem e metro fornecidas pela prefeitura, calcula a distância deste 
# para cada Cep único da cidade de SP, e retorna a distância, em metros, de cada um dos ceps em relação aos modais de transporte.
cep_calculo_transportes = CalculoDistanciaTransporte(ceps_sp, estacao_metro, ponto_onibus, estacao_trem, terminal_onibus).process()

# Aqui tenho a informação de cada um dos iptus registrados na prefeitura para o ano de 2023. Nesse caso, como nao posso mapear cada imovel
# com seu respectivo IPTU, irei usar a info agregada dos bairros. Para variaveis continuas, farei a média por CEP, para categóricas, primeiro
# realizarei o frequency encoding, que atribui para cada valor unico a quantidade de ocorrencias daquela categoria, e depois, soma todos os valores
# para cada cep.
iptu = AgregaIptuCep(iptu_estat_sp).process()

# Agrega os seguintes dados na base de imoveis para treino
# bo_distrito: proporção de ocorrências pelo total de habitantes
# cep_calculo_transportes: distância de cada cep em relação aos 4 modais de transporte mais próximos
# iptu: dados médios das características das edificações daquele cep
# estat_udh: dados sociais referentes a um determinado udh, atribuido a cada cep
imoveis_aplicacao = MesclaDadosMunicipais(dbimoveis, estat_udh, iptu, cep_calculo_transportes, bo_distrito, ceps_sp).process()

D:\Drive\Codigos bee6\bee6_notebooks\bee6_geral\aplica_dados_geo.py:201: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.cep.rename(columns={


### Aplica o modelo de machine learning para predizer preços dos imoveis da imobiliária

In [9]:
# Obter as colunas de features usadas pelo modelo
train_features = metadados.get('features', [])

# Selecionar apenas as colunas de features usadas pelo modelo
features_columns = [col for col in train_features if col in imoveis_aplicacao.columns]

# Preparar o DataFrame apenas com as colunas de features
X = imoveis_aplicacao[features_columns]

# Aplicar o modelo para fazer previsões
imoveis_aplicacao['predicao'] = xgb_model.predict(X)

# Criar o DataFrame final com 'id_imovel', 'preco_m2' e 'predicao'
result = imoveis_aplicacao[['id_imovel','cep','num_quartos','area_construida_m2', 'predicao']]

# Faz tratamentos na tabela resposta
result = result.query("id_imovel != 'Validar Dados Imóvel'")
result.drop_duplicates(subset = 'id_imovel', inplace = True)

# Exibir as primeiras linhas do DataFrame resultante para verificação
scores_ml_precificacao = result

# Cria a série temporal de preços de imóveis incorporando vendas urban space

In [20]:
# Ferramentas oferecidas
# Facebook prophet para séries históricas
# Dynamic pricing
# Mostrar muitas análises estatísticas

# Lasso e Ridge para séries temporais (ensemble dos 2)
# Value at Risk para cálculo de volatilidade
# Estudar conceitos de séries estacionárias com ARIMA ou SARIMA

### Prepara os dataframes

In [10]:
# Prepara todos os dataframes que serão usados para o calculo do FDI dinamico

imoveis_precificados = scores_ml_precificacao[['id_imovel','predicao','num_quartos']]
imoveis_precificados.columns  = ['id_imovel','m2_nov_23','num_quartos']
imoveis_vendidos = dbcompradores[['id_imovel','data_venda','valor_venda_m2']]
imoveis_vendidos.columns = ['id_imovel','data_venda','m2_valor_venda']

economia['data'] = pd.to_datetime(economia['data'])
fipezap_sp_ts['data'] = pd.to_datetime(fipezap_sp_ts['data'])
fipezap_sp_ts.columns = ['data','media_geral_fipezap','1q_fipezap','2q_fipezap','3q_fipezap','4q_fipezap']

# Dataframes que serão usados de fato
imoveis_totais = imoveis_precificados.merge(imoveis_vendidos, how = 'left', on = 'id_imovel')
economia_fipezap = economia.merge(fipezap_sp_ts, how = 'left', on = 'data').sort_values('data').set_index('data')
economia_fipezap.dropna(subset = 'media_geral_fipezap', inplace = True)

### Atribui aos imóveis suas respectivas séries históricas de acordo com numero de quartos

In [54]:
# Selecionar as variáveis exógenas da economia
variaveis_exogenas = ['igpm', 'selic', 'ipca', 'dolar', 'incc', 'ivgr', 'ipam', 'iiebr', 'cubsp']

# Converter todas as colunas relevantes para numérico
cols_to_convert = variaveis_exogenas + ['1q_fipezap', '2q_fipezap', '3q_fipezap', '4q_fipezap', 'media_geral_fipezap']
for col in cols_to_convert:
    economia_fipezap[col] = pd.to_numeric(economia_fipezap[col], errors='coerce')

# Preencher valores faltantes nas colunas selecionadas
economia_fipezap[cols_to_convert] = economia_fipezap[cols_to_convert].interpolate(method='linear')
economia_fipezap[cols_to_convert] = economia_fipezap[cols_to_convert].fillna(method='ffill')
economia_fipezap[cols_to_convert] = economia_fipezap[cols_to_convert].fillna(method='bfill')

# Mapear o número de quartos ao índice FipeZap correspondente, dependendo do numero de quartos, cada um dos imóveis terá uma projeção
# de preços diferente, em função da granularidade de preços do fipezap por quarto
def obter_indice_fipezap(num_quartos):
    if num_quartos == 1:
        return '1q_fipezap'
    elif num_quartos == 2:
        return '2q_fipezap'
    elif num_quartos == 3:
        return '3q_fipezap'
    elif num_quartos >= 4:
        return '4q_fipezap'
    else:
        return 'media_geral_fipezap'

imoveis_totais['indice_fipezap'] = imoveis_totais['num_quartos'].apply(obter_indice_fipezap)

# Dicionários para armazenar modelos e previsões
modelos = {}
previsoes = {}

for indice in imoveis_totais['indice_fipezap'].unique():
    # Extrai, para cada numero de quartos, a série temporal respectiva do fipezap e faz tratamentos
    serie_indice = economia_fipezap[indice].copy()
    serie_indice = serie_indice.dropna()
    serie_indice = serie_indice.astype(float)
    if not isinstance(serie_indice.index, pd.DatetimeIndex):
        serie_indice.index = pd.to_datetime(serie_indice.index)

    # Extrai as variaveis exogenas (economicas) do período e faz tratamentos
    exogenas = economia_fipezap.loc[serie_indice.index, variaveis_exogenas].copy()
    exogenas = exogenas.astype(float)
    serie_indice, exogenas = serie_indice.align(exogenas, join='inner')
    
    # Verificar e tratar valores faltantes
    if serie_indice.isnull().any():
        print(f"Valores faltantes em serie_indice para {indice}")
        serie_indice = serie_indice.fillna(method='ffill')
    if exogenas.isnull().any().any():
        print(f"Valores faltantes em exogenas para {indice}")
        exogenas = exogenas.fillna(method='ffill')
    
    # Ajustar o modelo SARIMAX
    try:
        modelo = SARIMAX(serie_indice, exog=exogenas, order=(1,1,1), seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False)
        resultado = modelo.fit(disp=False)
        modelos[indice] = resultado
    except ValueError as e:
        print(f"Erro ao ajustar o modelo para {indice}: {e}")
        continue
    
    # Passo 5: Fazer previsões para os próximos 60 meses
    # Criar um DateRange para as datas futuras, mantendo a frequência mensal
    ultima_data = serie_indice.index.max()
    datas_futuras = pd.date_range(start=ultima_data + pd.DateOffset(months=1), periods=60, freq='MS')
    
    # Criar um DataFrame com as variáveis exógenas futuras
    # Aqui, vamos supor que as variáveis exógenas futuras serão iguais ao último valor conhecido
    exogenas_futuras = pd.DataFrame(index=datas_futuras, columns=variaveis_exogenas)
    for col in variaveis_exogenas:
        ultimo_valor = exogenas[col].iloc[-1]
        exogenas_futuras[col] = ultimo_valor
    
    # Garantir que os tipos de dados estão corretos
    exogenas_futuras = exogenas_futuras.astype(float)
    
    # Fazer a previsão
    previsao = resultado.get_forecast(steps=60, exog=exogenas_futuras)
    previsao_media = previsao.predicted_mean
    
    # Armazenar as previsões
    previsoes[indice] = previsao_media

# Aplicar as previsões aos preços dos imóveis

# Lista para armazenar os DataFrames das previsões por imóvel
lista_previsoes_imoveis = []

for idx, row in imoveis_totais.iterrows():
    indice = row['indice_fipezap']
    if indice not in previsoes:
        print(f"Sem previsão disponível para o índice {indice}")
        continue

    # Recuperar a série histórica do índice até nov/23
    serie_historica = economia_fipezap.loc[:'2023-11-01', indice]
    serie_historica = serie_historica.dropna()
    serie_historica.index = pd.to_datetime(serie_historica.index)

    # Obter as previsões futuras
    previsao_indice = previsoes[indice]
    previsao_indice.index = pd.to_datetime(previsao_indice.index)

    # Combinar série histórica e previsões
    serie_completa = pd.concat([serie_historica, previsao_indice])

    # Calcular a variação percentual do índice a partir de nov/23
    indice_base = economia_fipezap.loc[pd.Timestamp('2023-11-01'), indice]

    # Verificar se indice_base não é nulo
    if pd.isna(indice_base):
        print(f"Valor base do índice {indice} em nov/23 está faltando.")
        continue

    var_percentual = serie_completa / indice_base

    # Aplicar a variação ao preço base do imóvel
    preco_base = row['m2_nov_23']
    serie_preco_projetado = preco_base * var_percentual

    # Criar um DataFrame com as previsões para o imóvel atual
    df_previsao_imovel = pd.DataFrame({
        'data': serie_preco_projetado.index,
        'preco_projetado_m2': serie_preco_projetado.values,
        'id_imovel': row['id_imovel']
    })

    # Adicionar outras colunas necessárias (se houver)
    # Por exemplo, calcular o preço projetado total
    if 'area_construida_m2' in row:
        area_construida = row['area_construida_m2']
        df_previsao_imovel['area_construida_m2'] = area_construida
        df_previsao_imovel['preco_projetado'] = df_previsao_imovel['preco_projetado_m2'] * area_construida

    # Definir a coluna 'data' como índice
    df_previsao_imovel.set_index('data', inplace=True)

    # Adicionar o DataFrame à lista
    lista_previsoes_imoveis.append(df_previsao_imovel)

# Combinar todas as previsões em um único DataFrame
df_previsoes_completo = pd.concat(lista_previsoes_imoveis)

# Resetar o índice para que 'data' seja uma coluna
df_previsoes_completo.reset_index(inplace=True)

# Concatenar todas as previsões por imóvel em um único DataFrame e adiciona o preço total (nao por m2)
dbPredicao = pd.concat(lista_previsoes_imoveis).reset_index()
area_imoveis = dbimoveis[['id_imovel','area_construida_m2']]
dbPredicao = dbPredicao.merge(area_imoveis, how = 'left', on = 'id_imovel')
dbPredicao['preco_projetado'] = dbPredicao['preco_projetado_m2'] * dbPredicao['area_construida_m2'] 
dbPredicao['id_imovel'].replace('NaN', np.nan, inplace=True)
dbPredicao = dbPredicao.dropna(subset=['id_imovel'])
dbPredicao = dbPredicao[['id_imovel','data','preco_projetado_m2','area_construida_m2','preco_projetado']]

C:\Users\guici\AppData\Local\Temp\ipykernel_28612\3354573526.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  economia_fipezap[cols_to_convert] = economia_fipezap[cols_to_convert].fillna(method='ffill')
C:\Users\guici\AppData\Local\Temp\ipykernel_28612\3354573526.py:12: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  economia_fipezap[cols_to_convert] = economia_fipezap[cols_to_convert].fillna(method='bfill')
c:\Users\guici\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\guici\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, s

# Cálculo de proximidade geográfica para cada imóvel

### Calcula a distância para cada modal de transporte

In [15]:
# Função para obter a distância do Google Distance Matrix API
def get_distance_google(origin, destination, mode='walking'):
    # origin e destination devem estar no formato 'lat,long'
    result = gmaps.distance_matrix(origins=[origin], destinations=[destination], mode=mode)
    
    # Verifica se a API retornou resultados válidos
    if result['rows'][0]['elements'][0]['status'] == 'OK':
        distance_meters = result['rows'][0]['elements'][0]['distance']['value']  # distância em metros
        return distance_meters
    else:
        return None

# Função para converter strings de lat/long em tuplas de floats
def converter_lat_long(lat_long_str):
    return tuple(map(float, lat_long_str.split(',')))

# Prepara as tabelas para serem mescladas
imoveis_prox = dbimoveis.dropna(subset = 'lat_long')[['id_imovel','lat_long']]
imoveis_prox.columns = ['id_imovel','lat_long_imovel']

terminal_onibus = terminal_onibus[['nome','endereco','lat_long']]
terminal_onibus.columns = ['nome_terminal','local_terminal','lat_long_terminal']

ponto_onibus = ponto_onibus[['nome','endereco','lat_long']]
ponto_onibus.columns = ['nome_ponto','local_ponto','lat_long_ponto']

estacao_trem = estacao_trem[['nome','linha','lat_long']]
estacao_trem.columns = ['estacao_trem','linha_trem','lat_long_trem']

estacao_metro = estacao_metro[['nome','linha','lat_long']]
estacao_metro.columns = ['estacao_metro','linha_metro','lat_long_metro']

# Aplicar a conversão nas colunas de lat/long
terminal_onibus['lat_long_terminal'] = terminal_onibus['lat_long_terminal'].apply(converter_lat_long)
ponto_onibus['lat_long_ponto'] = ponto_onibus['lat_long_ponto'].apply(converter_lat_long)
estacao_trem['lat_long_trem'] = estacao_trem['lat_long_trem'].apply(converter_lat_long)
estacao_metro['lat_long_metro'] = estacao_metro['lat_long_metro'].apply(converter_lat_long)
imoveis_prox['lat_long_imovel'] = imoveis_prox['lat_long_imovel'].apply(converter_lat_long)

# Construir KDTree para cada tipo de modal
tree_terminal_onibus = KDTree(terminal_onibus['lat_long_terminal'].tolist())
tree_ponto_onibus = KDTree(ponto_onibus['lat_long_ponto'].tolist())
tree_estacao_trem = KDTree(estacao_trem['lat_long_trem'].tolist())
tree_estacao_metro = KDTree(estacao_metro['lat_long_metro'].tolist())

# Encontrar os modais mais próximos para cada imóvel
resultados = []

for _, row in imoveis_prox.iterrows():
    lat_long_imovel = row['lat_long_imovel']

    # Encontrar o terminal de ônibus mais próximo usando KDTree
    dist_terminal_onibus, idx_terminal_onibus = tree_terminal_onibus.query(lat_long_imovel)
    lat_long_terminal = terminal_onibus.iloc[idx_terminal_onibus]['lat_long_terminal']
    nome_terminal = terminal_onibus.iloc[idx_terminal_onibus]['nome_terminal']
    local_terminal = terminal_onibus.iloc[idx_terminal_onibus]['local_terminal']
    
    # Encontrar o ponto de ônibus mais próximo usando KDTree
    dist_ponto_onibus, idx_ponto_onibus = tree_ponto_onibus.query(lat_long_imovel)
    lat_long_ponto = ponto_onibus.iloc[idx_ponto_onibus]['lat_long_ponto']
    nome_ponto = ponto_onibus.iloc[idx_ponto_onibus]['nome_ponto']
    local_ponto = ponto_onibus.iloc[idx_ponto_onibus]['local_ponto']

    # Encontrar a estação de trem mais próxima usando KDTree
    dist_estacao_trem, idx_estacao_trem = tree_estacao_trem.query(lat_long_imovel)
    lat_long_trem = estacao_trem.iloc[idx_estacao_trem]['lat_long_trem']
    estacao_trem_nome = estacao_trem.iloc[idx_estacao_trem]['estacao_trem']
    linha_trem = estacao_trem.iloc[idx_estacao_trem]['linha_trem']

    # Encontrar a estação de metrô mais próxima usando KDTree
    dist_estacao_metro, idx_estacao_metro = tree_estacao_metro.query(lat_long_imovel)
    lat_long_metro = estacao_metro.iloc[idx_estacao_metro]['lat_long_metro']
    estacao_metro_nome = estacao_metro.iloc[idx_estacao_metro]['estacao_metro']
    linha_metro = estacao_metro.iloc[idx_estacao_metro]['linha_metro']

    # Agora usar a API do Google para calcular a distância real (trajeto) para cada modal mais próximo

    # Terminal de ônibus mais próximo
    dist_terminal_onibus_real = get_distance_google(
        f"{lat_long_imovel[0]},{lat_long_imovel[1]}", 
        f"{lat_long_terminal[0]},{lat_long_terminal[1]}",
        mode='walking')

    # Ponto de ônibus mais próximo
    dist_ponto_onibus_real = get_distance_google(
        f"{lat_long_imovel[0]},{lat_long_imovel[1]}", 
        f"{lat_long_ponto[0]},{lat_long_ponto[1]}",
        mode='walking')

    # Estação de trem mais próxima
    dist_estacao_trem_real = get_distance_google(
        f"{lat_long_imovel[0]},{lat_long_imovel[1]}", 
        f"{lat_long_trem[0]},{lat_long_trem[1]}",
        mode='walking')

    # Estação de metrô mais próxima
    dist_estacao_metro_real = get_distance_google(
        f"{lat_long_imovel[0]},{lat_long_imovel[1]}", 
        f"{lat_long_metro[0]},{lat_long_metro[1]}",
        mode='walking')

    # Montar dicionário de resultados
    resultados.append({
        'id_imovel': row['id_imovel'],
        'lat_long_imovel': f"{lat_long_imovel[0]}, {lat_long_imovel[1]}",
        'nome_terminal': nome_terminal,
        'local_terminal': local_terminal,
        'lat_long_terminal': f"{lat_long_terminal[0]}, {lat_long_terminal[1]}",
        'dist_terminal(m)': dist_terminal_onibus_real,
        'nome_ponto': nome_ponto,
        'local_ponto': local_ponto,
        'lat_long_ponto': f"{lat_long_ponto[0]}, {lat_long_ponto[1]}",
        'dist_ponto(m)': dist_ponto_onibus_real,
        'estacao_trem': estacao_trem_nome,
        'linha_trem': linha_trem,
        'lat_long_trem': f"{lat_long_trem[0]}, {lat_long_trem[1]}",
        'dist_trem(m)': dist_estacao_trem_real,
        'estacao_metro': estacao_metro_nome,
        'linha_metro': linha_metro,
        'lat_long_metro': f"{lat_long_metro[0]}, {lat_long_metro[1]}",
        'dist_metro(m)': dist_estacao_metro_real
    })

# Criar DataFrame final
df_resultados = pd.DataFrame(resultados)

# Organizar as colunas na ordem desejada
dbModais = df_resultados[[
    'id_imovel', 'lat_long_imovel', 'nome_terminal', 'local_terminal', 'lat_long_terminal', 'dist_terminal(m)',
    'nome_ponto', 'local_ponto', 'lat_long_ponto', 'dist_ponto(m)', 'estacao_trem', 'linha_trem', 'lat_long_trem', 'dist_trem(m)',
    'estacao_metro', 'linha_metro', 'lat_long_metro', 'dist_metro(m)']]

C:\Users\guici\AppData\Local\Temp\ipykernel_12600\3324493197.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  terminal_onibus['lat_long_terminal'] = terminal_onibus['lat_long_terminal'].apply(converter_lat_long)
C:\Users\guici\AppData\Local\Temp\ipykernel_12600\3324493197.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ponto_onibus['lat_long_ponto'] = ponto_onibus['lat_long_ponto'].apply(converter_lat_long)
C:\Users\guici\AppData\Local\Temp\ipykernel_12600\3324493197.py:36: SettingWithCopyWarning: 

### Calcula a distância para cada POI   

In [16]:
def processar_imoveis_e_pois(df_imoveis_prox, api_key, tipos_estabelecimento, parametros_extras, raio):
    
    # Lista para armazenar os resultados de cada imóvel
    lista_resultados = []

    # Iterar sobre cada linha do DataFrame de imóveis
    for _, row in df_imoveis_prox.iterrows():
        id_imovel = row['id_imovel']
        lat_long_imovel = row['lat_long_imovel']
        
        # Verificar se lat_long_imovel é uma tupla ou coluna com lat e lng separadas
        if isinstance(lat_long_imovel, tuple):
            lat, lng = lat_long_imovel
        elif isinstance(lat_long_imovel, str):
            lat, lng = map(float, lat_long_imovel.split(","))
        else:
            raise ValueError("O valor de 'lat_long_imovel' deve ser uma string 'lat,lng' ou uma tupla (lat, lng).")
        
        # Iterar sobre cada tipo de estabelecimento
        for tipo in tipos_estabelecimento:
            url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
            params = {
                'location': f'{lat},{lng}',
                'radius': raio,
                'type': tipo,
                'key': api_key,
                'fields': ','.join(parametros_extras) }
            
            # Fazer a requisição
            response = requests.get(url, params=params)
            
            # Se a requisição foi bem-sucedida
            if response.status_code == 200:
                dados = response.json()
                if dados.get('status') == 'OK':
                    for resultado in dados.get('results', []):
                        lat_poi = resultado['geometry']['location']['lat']
                        lng_poi = resultado['geometry']['location']['lng']
                        
                        poi = {
                            'id_imovel': id_imovel,
                            'lat_long_imovel': f"{lat},{lng}",
                            'nome_poi': resultado.get('name'),
                            'endereco_poi': resultado.get('vicinity'),
                            'lat_long_poi': f"{lat_poi},{lng_poi}",
                            'tipo_poi': tipo}

                        # Adicionar os parâmetros extras definidos externamente
                        for parametro in parametros_extras:
                            poi[parametro] = resultado.get(parametro, None)
                        
                        lista_resultados.append(poi)

    # Converter a lista de resultados em um DataFrame
    df_resultados = pd.DataFrame(lista_resultados)
    
    return df_resultados

# Exemplo de uso
if __name__ == "__main__":

    # Lista de tipos de estabelecimento a buscar
    tipos_estabelecimento = ['restaurant', 'bar','pharmacy','bakery','supermarket','school','gym','park']
    
    # Lista de parâmetros extras a serem extraídos para cada POI
    parametros_extras = ['rating'] 
    
    # Definir o raio de busca em metros
    raio_busca = 500  
    
    # Processar os imóveis e obter os POIs
    dbPOIs = processar_imoveis_e_pois(imoveis_prox, chave_api, tipos_estabelecimento, parametros_extras, raio_busca)

# Faz o upload de tudo no google spreadsheet

In [56]:
def converte_numeros_brasil(df):
    """Formats numeric columns of a DataFrame to Brazilian number format without thousands separator."""
    # Format the date column if it exists
    if 'data' in df.columns:
        df['data'] = pd.to_datetime(df['data']).dt.strftime('%d/%m/%Y')
    
    # Identify numeric columns (integers and floats)
    numeric_cols = df.select_dtypes(include=['int', 'float']).columns
    
    # Define the formatting function
    def format_number(x):
        if pd.isnull(x):
            return ''
        return '{:.2f}'.format(x).replace('.', ',')
    
    # Apply the formatting function to each numeric column
    for col in numeric_cols:
        df[col] = df[col].apply(format_number)
    
    return df

def get_creds():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_PATH, SCOPES)
            creds = flow.run_console()
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

def dataframe_to_sheet(service, spreadsheet_id, dataframe, sheet_name):
    dataframe = dataframe.astype(str)
    values = [dataframe.columns.tolist()] + dataframe.values.tolist()
    body = {'values': values}
    range_name = f"{sheet_name}!A1"
    service.spreadsheets().values().clear(
        spreadsheetId=spreadsheet_id, range=range_name).execute()
    service.spreadsheets().values().update(
        spreadsheetId=spreadsheet_id,
        range=range_name,
        valueInputOption="RAW",
        body=body).execute()

def upload_dataframes_to_sheets(dataframes, sheet_names):
    creds = get_creds()
    service = build('sheets', 'v4', credentials=creds)
    for dataframe, sheet_name in zip(dataframes, sheet_names):
        dataframe_to_sheet(service, SAMPLE_SPREADSHEET_ID, dataframe, sheet_name)

# Use the function on your DataFrame
dbPredicao = converte_numeros_brasil(dbPredicao)

# Upload multiple dataframes
upload_dataframes_to_sheets([dbPredicao], ['dbPredicao'])